### OpenAlex --> Gephi Pipeline

To run `networkx`, `pandas`, `requests` and `openpyxl` need to be installed, they are all available on conda: `conda install -c conda-forge pandas requests networkx openpyxl`

In [4]:
import csv
import os
import sys
import time
from pathlib import Path
import pandas as pd
import requests
import networkx as nx
from networkx.algorithms.community import louvain_communities

`EXCEL_PATH` and `EXCLUDED_SHEET` specify where we can find the topics we manually identified as not being relevant to this study. `SEARCH_QUERY` is the query used to search OpenAlex.

In [9]:
EXCEL_PATH = Path(
    "../data/openAlex/sonification_audifcation_sonify_sonifying_sonified_"
    "auditoryGraph_auditoryDisplay_informationThroughSound.xlsx"
)
EXCLUDED_SHEET = "Excluded_Terms"
 
SEARCH_QUERY = (
    'Sonification OR Audification OR Sonify OR Sonifying OR Sonified '
    'OR "Auditory Graph" OR "Auditory Display" OR "Information Through Sound"'
) 

Variables for output, API calls, and search terms.

In [10]:
OUT_DIR = Path.cwd()
PUBS_CSV = OUT_DIR / "publications_kept.csv"
EDGES_CSV = OUT_DIR / "pub2ref.csv"
EDGES_TRIM_CSV = OUT_DIR / "pub2ref_remove_unique.csv"
GEXF_PATH = OUT_DIR / "citation_network.gexf"
 
API_KEY = "8Tb0QywKVVtbUVWmxwY4dp"
BASE_URL = "https://api.openalex.org/works"
PER_PAGE = 200
MAX_RETRIES = 6
REQUEST_TIMEOUT = 60
SELECT_FIELDS = ("id,doi,title,display_name,publication_year,type,"
                 "cited_by_count,primary_topic,primary_location,referenced_works")

Functions to keep things neater.

In [11]:
def norm(s):
    return " ".join(str(s).strip().casefold().split())
 
def short_id(openalex_id):
    return (openalex_id or "").rstrip("/").split("/")[-1]
 
def load_excluded_topics(xlsx_path, sheet):
    df = pd.read_excel(xlsx_path, sheet_name=sheet, header=None)
    terms = set()
    for col in df.columns:
        for value in df[col].dropna().tolist():
            text = str(value).strip()
            if text and text.lower() != "nan":
                terms.add(norm(text))
    return terms
 
def api_get(session, params):
    params = {**params, "api_key": API_KEY}
    for attempt in range(1, MAX_RETRIES + 1):
        resp = session.get(BASE_URL, params=params, timeout=REQUEST_TIMEOUT)
        if resp.status_code == 200:
            return resp.json()
        if resp.status_code == 429:
            remaining = resp.headers.get("X-RateLimit-Remaining-USD")
            if (remaining is not None and remaining not in ("", None)
                    and float(remaining) <= 0) or "budget" in resp.text.lower():
                raise RuntimeError("OpenAlex daily budget exhausted: " + resp.text[:300])
            time.sleep(min(2 ** attempt, 30))
            continue
        if resp.status_code in (500, 502, 503, 504):
            time.sleep(min(2 ** attempt, 30))
            continue
        raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:300]}")
    raise RuntimeError("Giving up after retries")
 
def primary_source_name(work):
    return (((work.get("primary_location") or {}).get("source") or {})
            .get("display_name") or "")
 
def primary_topic_name(work):
    return (work.get("primary_topic") or {}).get("display_name") or ""
 
def harvest(session, excluded):
    params = {"search": SEARCH_QUERY, "per_page": PER_PAGE,
              "select": SELECT_FIELDS, "cursor": "*"}
    kept = []
    while params["cursor"]:
        page = api_get(session, params)
        for work in page["results"]:
            topic = primary_topic_name(work)
            if topic and norm(topic) in excluded:
                continue
            if norm(primary_source_name(work)) == "zenodo":
                continue
            kept.append(work)
        params["cursor"] = page["meta"].get("next_cursor")
    return kept
 
def build_edges(kept):
    rows = []
    for work in kept:
        pub = short_id(work["id"])
        for ref in (work.get("referenced_works") or []):
            rows.append((pub, short_id(ref)))
    return pd.DataFrame(rows, columns=["publication_id", "reference_id"])
 
def remove_single_reference_edges(edges):
    return edges[edges.duplicated(subset=["reference_id"], keep=False)].copy()
 
def build_graph(edges, corpus_meta):
    G = nx.DiGraph()
    G.add_edges_from(edges[["publication_id", "reference_id"]].itertuples(
        index=False, name=None))
    for node in G.nodes():
        meta = corpus_meta.get(node, {})
        in_corpus = node in corpus_meta
        attrs = {
            "label": meta.get("title") or node,
            "kind": "publication" if in_corpus else "reference_only",
            "in_corpus": bool(in_corpus),
            "cited_by_within_corpus": int(G.in_degree(node)),
            "references_made": int(G.out_degree(node)),
        }
        if meta.get("year") is not None:
            try:
                attrs["year"] = int(meta["year"])
            except (TypeError, ValueError):
                pass
        for key in ("source", "type", "primary_topic", "subfield", "field", "domain", "doi"):
            if meta.get(key):
                attrs[key] = str(meta[key])
        G.nodes[node].update(attrs)
    return G
 
def add_communities(G):
    for cid, members in enumerate(louvain_communities(G.to_undirected(), seed=42)):
        for node in members:
            G.nodes[node]["modularity_class"] = int(cid)

In [13]:
excluded = load_excluded_topics(EXCEL_PATH, EXCLUDED_SHEET)
session = requests.Session()
kept = harvest(session, excluded)

corpus_meta = {}
with PUBS_CSV.open("w", newline="", encoding="utf-8-sig") as cf:
    writer = None
    for w in kept:
        pt = w.get("primary_topic") or {}
        ploc = w.get("primary_location") or {}
        sid = short_id(w["id"])
        corpus_meta[sid] = {
            "title": w.get("title") or w.get("display_name") or "",
            "year": w.get("publication_year"),
            "source": (ploc.get("source") or {}).get("display_name") or "",
            "type": w.get("type") or "",
            "primary_topic": pt.get("display_name", ""),
            "subfield": (pt.get("subfield") or {}).get("display_name", ""),
            "field": (pt.get("field") or {}).get("display_name", ""),
            "domain": (pt.get("domain") or {}).get("display_name", ""),
            "doi": w.get("doi") or "",
        }
        row = {"openalex_id": sid, **corpus_meta[sid],
               "cited_by_count": w.get("cited_by_count", ""),
               "num_referenced_works": len(w.get("referenced_works") or [])}
        if writer is None:
            writer = csv.DictWriter(cf, fieldnames=list(row.keys()))
            writer.writeheader()
        writer.writerow(row)

edges = build_edges(kept)
edges.to_csv(EDGES_CSV, index=False)

edges_final = remove_single_reference_edges(edges)
edges_final.to_csv(EDGES_TRIM_CSV, index=False)

G = build_graph(edges_final, corpus_meta)
if G.number_of_nodes():
    add_communities(G)
nx.write_gexf(G, GEXF_PATH)

SyntaxError: 'return' outside function (3602271609.py, line 41)